# Example 01.08 — run performance monitoring

Joins all predictions with actuals and computes full-scope regression metrics.
Only bounded report metadata is returned to the notebook.


In [1]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ml_app_client").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Start Jupyter inside the ml-app repository")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from ml_app_client import MLAppClient, ResourceNotFoundError

client = MLAppClient.connect()
print("Connected to ML App")


Connected to ML App


## Choose resource names

These are normal names passed to `ml_app_client`. Edit them directly and use the
same values in the following notebooks. Dataset and pipeline names are resolved
inside the selected Business Case. Business Case and service names are globally
unique on one ML App installation.


In [2]:
# These are ordinary platform resource names. Change the two globally unique
# names (Business Case and model service) when sharing one installation.
BUSINESS_CASE_NAME = "[MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo"
TRAINING_DATASET_NAME = "Example01 Estates - Training"
SCORING_DATASET_NAME = "Example01 Estates - Batch Input"
ACTUALS_DATASET_NAME = "Example01 Estates - Actuals"
TRAINING_PIPELINE_NAME = "Example01 03 - AutoML Training"
BATCH_PIPELINE_NAME = "Example01 05 - Batch Scoring"
MONITORING_PIPELINE_NAME = "Example01 07 - Performance Monitoring"
MODEL_NAME = "Example01 Estates Price Model"
OUTPUT_NAME_PREFIX = "Example01 Estates AutoML"
MODEL_SERVICE_NAME = "Example01 10 - Estates Model Service - demo"

TRAINING_RUN_KEY = "Example01-training-v2"
BATCH_RUN_KEY = "Example01-batch-scoring-v2"
MONITORING_RUN_KEY = "Example01-monitoring-v2"

print("Resource names configured for:", BUSINESS_CASE_NAME)


Resource names configured for: [MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo


In [3]:
try:
    run = client.pipeline_run_by_operation_key(business_case_name=BUSINESS_CASE_NAME, pipeline_name=MONITORING_PIPELINE_NAME, operation_key=MONITORING_RUN_KEY)
    started = False
except ResourceNotFoundError:
    run = client.run_pipeline_by_name(business_case_name=BUSINESS_CASE_NAME, pipeline_name=MONITORING_PIPELINE_NAME, runtime_parameters={"client_operation_key": MONITORING_RUN_KEY})
    started = True
print(f"{'STARTED' if started else 'REUSED'} run {run.id}; status={run.status}")
finished = client.wait_for_pipeline_run(run, timeout=3600)
report = client.scoring_report_for_run(
    finished,
    business_case_name=BUSINESS_CASE_NAME,
)
metrics = {
    metric["id"]: metric["value"]
    for metric in report["evaluation"].get("metrics", [])
}
print({
    "run_id": finished.id,
    "processed_rows": finished.processed_row_count,
    "evaluated_rows": report["evaluated_row_count"],
    "report_id": report["id"],
    "metrics": metrics,
})


STARTED run 4d3c1189-c278-4360-a2b1-92b272f62b66; status=queued
{'run_id': '4d3c1189-c278-4360-a2b1-92b272f62b66', 'processed_rows': 100000, 'evaluated_rows': 100000, 'report_id': '4af96127-ea26-42d7-9869-c7db66ad2146', 'metrics': {'mae': 89366.75576140395, 'rmse': 132883.9937168274, 'r2': 0.9495815291675163, 'mean_residual': -68711.74205471313, 'mape': 0.08587507966109952}}
